# Quantum Attention — AG News (Topic Classification, 4 classes)

Single-head and multi-head **quantum-scored attention** vs classical dot-product
attention, with a separability-grounded **entanglement ablation**
(`none / intra / cross / full`), on **AGNEWS**.

This notebook runs four experiments and **checkpoints every result to Google Drive**
after each (condition, seed) run, so it survives Colab disconnects — just re-run the
cell and it resumes where it stopped.

**Experiments**
1. **A — Architecture ablation (4 heads):** quantum head as 1 of 4; the realistic setting.
2. **B — Single-head ablation:** quantum head is the *only* attention → tests whether
   entanglement matters once *undiluted*.
3. **C — Data-efficiency:** accuracy vs training size (the low-data regime where quantum
   methods are most defensible).
4. **D — Qubit width / encoding bottleneck:** 2 vs 4 vs 8 qubits (1/2/4 angles per side)
   → tests whether a wider encoding lets entanglement matter.

**Run order:** Setup → Download → Core → Config → Harness → run any Experiment cell(s)
→ Reporting. Recommended: **A and B first** (fast, answer the core question), then C and D.

**Note:** AG News texts are long (p95≈53 tokens), so `MAX_LEN=50` and the quantum runs are the slowest of the three datasets. Lower `MAX_LEN` to 30 to trade a little accuracy for substantial speed.


## 1 · Setup (install, mount Drive, choose device)

In [ ]:
# PennyLane is the quantum simulator; torch ships with Colab.
!pip -q install -U pennylane
import os, json, time, math, itertools
import torch
print("torch", torch.__version__)
import pennylane as qml; print("pennylane", qml.__version__)

# Mount Google Drive for checkpointing.
from google.colab import drive
drive.mount('/content/drive')

DATASET = 'agnews'
DRIVE_ROOT = '/content/drive/MyDrive/quantum_attention'
RESULTS_DIR = os.path.join(DRIVE_ROOT, DATASET)
FIG_DIR = os.path.join(RESULTS_DIR, 'figures')
os.makedirs(RESULTS_DIR, exist_ok=True); os.makedirs(FIG_DIR, exist_ok=True)
print("Results ->", RESULTS_DIR)

# The 4-qubit statevector sim is CPU-bound; GPU does not accelerate default.qubit.
# CPU is the right choice and avoids device-mismatch. (High-RAM runtime helps memory.)
DEVICE = 'cpu'
print("device =", DEVICE)


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 105.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 937.5/937.5 kB 41.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.5/25.5 MB 72.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 66.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.2/167.2 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 100.4 MB/s eta 0:00:00
torch 2.11.0+cpu
pennylane 0.45.1
Mounted at /content/drive
Results -> /content/drive/MyDrive/quantum_attention/agnews
device = cpu


## 2 · Download data (cached on the Colab VM)

In [ ]:
import urllib.request
def download(url, path):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    if os.path.exists(path) and os.path.getsize(path) > 0:
        print("exists:", path); return
    urllib.request.urlretrieve(url, path)
    print("downloaded:", path, os.path.getsize(path), "bytes")

download("https://raw.githubusercontent.com/mhjabreel/CharCnn_Keras/master/data/ag_news_csv/train.csv", "agnews/train.csv")
download("https://raw.githubusercontent.com/mhjabreel/CharCnn_Keras/master/data/ag_news_csv/test.csv", "agnews/test.csv")


downloaded: agnews/train.csv 29470338 bytes
downloaded: agnews/test.csv 1857427 bytes


## 3 · Core code (model + utilities)
Generalized quantum attention head (configurable qubit count), the small Transformer, data utilities, val-based training, and the permutation test.

In [ ]:
"""
qatt_core.py — Consolidated core for the Quantum Attention experiments.
Embedded verbatim into each Colab notebook. Generalized to support a
configurable number of qubits (even; half encode the query, half the key).

Contents:
  * QuantumAttentionHead / MixedMultiHeadAttention / TransformerBlock / TextTransformer
  * attention_entropy
  * tokenize / build_vocab / batchify
  * run_one (val-based model selection) + permutation_test + mean_std
The dataset loaders live in the notebook (they differ per dataset).
"""

import math, time, itertools
import torch
import torch.nn as nn
import torch.nn.functional as F
import pennylane as qml

PAD, UNK = 0, 1


# =========================================================================== #
# Quantum scoring circuit (generalized to n_qubits, half query / half key)
# =========================================================================== #
def build_quantum_score_qnode(n_qubits=4, entangle="full"):
    """Wires 0..h-1 carry query angles, wires h..2h-1 carry key angles (h=n_qubits/2).
    weights shape (2, n_qubits, 3): rotation layer 0 (pre-entangler) + layer 1 (post).
    The post-entangler rotation layer is REQUIRED so CZ gates (diagonal in Z) become
    visible to the Z-basis readout; without it entanglement has zero effect.

    entangle:
      none  -> no CZ                          (score separable: <Z_q0 Z_k0>=<Z_q0><Z_k0>)
      intra -> CZ within query / within key   (still separable for this readout)
      cross -> CZ pairing q_i with k_i         (genuine query<->key entanglement)
      full  -> cross + intra
    """
    assert n_qubits % 2 == 0 and n_qubits >= 2
    h = n_qubits // 2
    q_wires = list(range(h))
    k_wires = list(range(h, n_qubits))
    cross = [(q_wires[i], k_wires[i]) for i in range(h)]
    intra = ([(q_wires[i], q_wires[i + 1]) for i in range(h - 1)]
             + [(k_wires[i], k_wires[i + 1]) for i in range(h - 1)])
    cz_pairs = {"none": [], "intra": intra,
                "cross": cross, "full": cross + intra}[entangle]
    dev = qml.device("default.qubit", wires=n_qubits)

    @qml.qnode(dev, interface="torch", diff_method="backprop")
    def circuit(inputs, weights):
        qml.AngleEmbedding(inputs, wires=range(n_qubits), rotation="Y")
        for w in range(n_qubits):
            qml.RX(weights[0, w, 0], wires=w)
            qml.RY(weights[0, w, 1], wires=w)
            qml.RZ(weights[0, w, 2], wires=w)
        for a, b in cz_pairs:
            qml.CZ(wires=[a, b])
        for w in range(n_qubits):
            qml.RX(weights[1, w, 0], wires=w)
            qml.RY(weights[1, w, 1], wires=w)
            qml.RZ(weights[1, w, 2], wires=w)
        return qml.expval(qml.PauliZ(q_wires[0]) @ qml.PauliZ(k_wires[0]))

    return circuit


class QuantumAttentionHead(nn.Module):
    def __init__(self, head_dim, n_qubits=4, entangle="full", chunk=8192):
        super().__init__()
        assert n_qubits % 2 == 0
        self.n_qubits = n_qubits
        self.h = n_qubits // 2                       # angles encoded per side
        self.entangle = entangle
        self.chunk = chunk                           # max pairs per circuit call
        self.q_proj = nn.Linear(head_dim, self.h)
        self.k_proj = nn.Linear(head_dim, self.h)
        self.qweights = nn.Parameter(0.1 * torch.randn(2, n_qubits, 3))
        self.scale = nn.Parameter(torch.tensor(4.0))
        self.circuit = build_quantum_score_qnode(n_qubits, entangle)

    def _run_circuit(self, flat, qw):
        # Evaluate in chunks so peak memory stays bounded (matters at n_qubits>=6,
        # where the statevector is 2^n_qubits per pair). Each chunk is autograd-
        # connected, so gradients still flow and accumulate into qw.
        n = flat.shape[0]
        if n <= self.chunk:
            return self.circuit(flat, qw)
        outs = [self.circuit(flat[i:i + self.chunk], qw)
                for i in range(0, n, self.chunk)]
        return torch.cat(outs, dim=0)

    def forward(self, q, k, v, key_mask=None, debug=False):
        B, L, D = q.shape
        qa = torch.tanh(self.q_proj(q)) * math.pi     # (B,L,h)
        ka = torch.tanh(self.k_proj(k)) * math.pi
        qa_exp = qa.unsqueeze(2).expand(B, L, L, self.h)
        ka_exp = ka.unsqueeze(1).expand(B, L, L, self.h)
        pairs = torch.cat([qa_exp, ka_exp], dim=-1)   # (B,L,L,n_qubits)
        flat = pairs.reshape(-1, self.n_qubits).float()
        # default.qubit simulates on CPU; run there and move scalars back.
        scores = self._run_circuit(flat.cpu(), self.qweights.cpu())
        scores = scores.reshape(B, L, L).to(device=v.device, dtype=v.dtype)
        if debug:
            with torch.no_grad():
                print(f"[qhead] score range [{scores.min():.3f},{scores.max():.3f}] "
                      f"std={scores.std():.3f}")
        scores = scores * self.scale
        if key_mask is not None:
            scores = scores.masked_fill(key_mask == 0, float("-inf"))
        attn = F.softmax(scores, dim=-1)
        out = torch.matmul(attn, v)
        return out, attn

    def self_test(self, device="cpu"):
        x = torch.randn(2, 3, self.q_proj.in_features, device=device)
        self.to(device)
        out, attn = self.forward(x, x, x, debug=True)
        out.sum().backward()
        g = self.qweights.grad
        gn = None if g is None else g.norm().item()
        ok = gn and gn > 1e-8
        print(f"[self_test] nq={self.n_qubits} grad_norm={gn} "
              f"{'OK' if ok else 'DEAD GRADIENT'}")
        self.zero_grad()
        return ok


class MixedMultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads, use_quantum=True, entangle="full", n_qubits=4):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_model, self.n_heads = d_model, n_heads
        self.head_dim = d_model // n_heads
        self.use_quantum = use_quantum
        self.q_lin = nn.Linear(d_model, d_model)
        self.k_lin = nn.Linear(d_model, d_model)
        self.v_lin = nn.Linear(d_model, d_model)
        self.out_lin = nn.Linear(d_model, d_model)
        if use_quantum:
            self.qhead = QuantumAttentionHead(self.head_dim, n_qubits, entangle)

    def forward(self, x, key_mask=None, return_attn=False, debug=False):
        B, L, _ = x.shape
        H, Dh = self.n_heads, self.head_dim
        q = self.q_lin(x).view(B, L, H, Dh).transpose(1, 2)
        k = self.k_lin(x).view(B, L, H, Dh).transpose(1, 2)
        v = self.v_lin(x).view(B, L, H, Dh).transpose(1, 2)
        outs, attns = [], {}
        for hh in range(H):
            if self.use_quantum and hh == 0:
                o, a = self.qhead(q[:, hh], k[:, hh], v[:, hh],
                                  key_mask=key_mask, debug=debug)
                attns["quantum"] = a
            else:
                sc = torch.matmul(q[:, hh], k[:, hh].transpose(-2, -1)) / math.sqrt(Dh)
                if key_mask is not None:
                    sc = sc.masked_fill(key_mask == 0, float("-inf"))
                a = F.softmax(sc, dim=-1)
                o = torch.matmul(a, v[:, hh])
                if hh == 1 or (not self.use_quantum and hh == 0):
                    attns.setdefault("classical", a)
            outs.append(o)
        out = torch.stack(outs, dim=1).transpose(1, 2).reshape(B, L, H * Dh)
        out = self.out_lin(out)
        return (out, attns) if return_attn else out


class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, ff_dim, use_quantum=False,
                 entangle="full", n_qubits=4, dropout=0.1):
        super().__init__()
        self.attn = MixedMultiHeadAttention(d_model, n_heads, use_quantum,
                                            entangle, n_qubits)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(nn.Linear(d_model, ff_dim), nn.GELU(),
                                nn.Linear(ff_dim, d_model))
        self.drop = nn.Dropout(dropout)

    def forward(self, x, key_mask=None, return_attn=False, debug=False):
        if return_attn:
            a_out, attns = self.attn(x, key_mask, return_attn=True, debug=debug)
        else:
            a_out, attns = self.attn(x, key_mask, debug=debug), None
        x = self.norm1(x + self.drop(a_out))
        x = self.norm2(x + self.drop(self.ff(x)))
        return (x, attns) if return_attn else x


class TextTransformer(nn.Module):
    def __init__(self, vocab_size, n_classes, d_model=64, n_heads=4, ff_dim=128,
                 max_len=40, use_quantum=True, entangle="full", n_qubits=4,
                 quantum_in_block=0, dropout=0.1):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, d_model, padding_idx=0)
        self.pos_emb = nn.Embedding(max_len, d_model)
        self.max_len = max_len
        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, n_heads, ff_dim,
                             use_quantum=(use_quantum and b == quantum_in_block),
                             entangle=entangle, n_qubits=n_qubits, dropout=dropout)
            for b in range(2)])
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, n_classes)

    def forward(self, input_ids, attn_mask, return_attn=False, debug=False):
        B, L = input_ids.shape
        L = min(L, self.max_len)
        input_ids = input_ids[:, :L]
        attn_mask = attn_mask[:, :L]
        pos = torch.arange(L, device=input_ids.device).unsqueeze(0).expand(B, L)
        x = self.tok_emb(input_ids) + self.pos_emb(pos)
        key_mask = attn_mask.unsqueeze(1)
        collected = {}
        for bi, blk in enumerate(self.blocks):
            if return_attn:
                x, attns = blk(x, key_mask, return_attn=True, debug=(debug and bi == 0))
                if attns:
                    collected[f"block{bi}"] = attns
            else:
                x = blk(x, key_mask, debug=(debug and bi == 0))
        x = self.norm(x)
        m = attn_mask.unsqueeze(-1).float()
        pooled = (x * m).sum(1) / m.sum(1).clamp(min=1.0)
        logits = self.head(pooled)
        return (logits, collected) if return_attn else logits


def attention_entropy(attn, key_mask=None, eps=1e-9):
    p = attn.clamp(min=eps)
    ent = -(p * p.log()).sum(-1)
    if key_mask is not None:
        valid = key_mask.squeeze(1).float()
        return ((ent * valid).sum() / valid.sum().clamp(min=1.0)).item()
    return ent.mean().item()


# =========================================================================== #
# Data utilities
# =========================================================================== #
def tokenize(s):
    return s.lower().split()


def build_vocab(examples, min_freq=2):
    from collections import Counter
    c = Counter()
    for e in examples:
        c.update(tokenize(e["text"]))
    v = {"<pad>": PAD, "<unk>": UNK}
    for w, f in c.most_common():
        if f >= min_freq:
            v[w] = len(v)
    return v


def encode(s, v, ml):
    return [v.get(t, UNK) for t in tokenize(s)][:ml]


def batchify(examples, v, ml, bs, shuffle, seed, device):
    idx = list(range(len(examples)))
    if shuffle:
        g = torch.Generator().manual_seed(seed)
        idx = torch.randperm(len(examples), generator=g).tolist()
    batches = []
    for i in range(0, len(examples), bs):
        chunk = [examples[j] for j in idx[i:i + bs]]
        seqs = [encode(e["text"], v, ml) for e in chunk]
        L = max(max((len(s) for s in seqs), default=1), 1)
        ids = torch.full((len(seqs), L), PAD, dtype=torch.long)
        m = torch.zeros((len(seqs), L), dtype=torch.long)
        for r, s in enumerate(seqs):
            if s:
                ids[r, :len(s)] = torch.tensor(s)
                m[r, :len(s)] = 1
        y = torch.tensor([e["label"] for e in chunk])
        batches.append((ids.to(device), m.to(device), y.to(device)))
    return batches


@torch.no_grad()
def accuracy(model, batches):
    model.eval()
    c = t = 0
    for ids, m, y in batches:
        c += (model(ids, m).argmax(-1) == y).sum().item()
        t += y.numel()
    return c / max(t, 1)


@torch.no_grad()
def mean_entropy(model, batches, max_batches=6):
    model.eval()
    q, cl = [], []
    for bi, (ids, m, y) in enumerate(batches):
        if bi >= max_batches:
            break
        _, attns = model(ids, m, return_attn=True)
        b0 = attns.get("block0", {})
        km = m[:, :ids.shape[1]].unsqueeze(1)
        if "quantum" in b0:
            q.append(attention_entropy(b0["quantum"], km))
        if "classical" in b0:
            cl.append(attention_entropy(b0["classical"], km))
    return (sum(q) / len(q) if q else float("nan"),
            sum(cl) / len(cl) if cl else float("nan"))


# =========================================================================== #
# Train one (condition, seed) with val-based model selection
# =========================================================================== #
CONDITIONS = {
    "classical": dict(use_quantum=False, entangle="full"),
    "qnone":     dict(use_quantum=True,  entangle="none"),
    "qintra":    dict(use_quantum=True,  entangle="intra"),
    "qcross":    dict(use_quantum=True,  entangle="cross"),
    "qfull":     dict(use_quantum=True,  entangle="full"),
}


def run_one(cond, seed, data, vocab, n_classes, cfg):
    """cfg: dict with epochs, max_len, batch_size, d_model, n_heads, n_qubits,
    lr, qlr, device."""
    train_fit, val, test = data
    cc = CONDITIONS[cond]
    device = cfg["device"]
    torch.manual_seed(seed)
    model = TextTransformer(
        vocab_size=len(vocab), n_classes=n_classes,
        d_model=cfg["d_model"], n_heads=cfg["n_heads"], max_len=cfg["max_len"],
        use_quantum=cc["use_quantum"], entangle=cc["entangle"],
        n_qubits=cfg["n_qubits"]).to(device)
    qp = [p for n, p in model.named_parameters() if "qweights" in n]
    cp = [p for n, p in model.named_parameters() if "qweights" not in n]
    groups = [{"params": cp, "lr": cfg["lr"]}]
    if qp:
        groups.append({"params": qp, "lr": cfg["qlr"]})
    opt = torch.optim.Adam(groups)
    val_b = batchify(val, vocab, cfg["max_len"], 128, False, 0, device)
    test_b = batchify(test, vocab, cfg["max_len"], 128, False, 0, device)
    best_val, test_at_best, best_ep = -1.0, 0.0, 0
    t0 = time.time()
    for ep in range(1, cfg["epochs"] + 1):
        model.train()
        for ids, m, y in batchify(train_fit, vocab, cfg["max_len"],
                                  cfg["batch_size"], True, 1000 * seed + ep, device):
            opt.zero_grad()
            F.cross_entropy(model(ids, m), y).backward()
            opt.step()
        va = accuracy(model, val_b)
        if va > best_val:
            best_val, test_at_best, best_ep = va, accuracy(model, test_b), ep
    h_q, h_c = mean_entropy(model, test_b)
    return dict(test_acc=test_at_best, val_acc=best_val, best_epoch=best_ep,
                H_quantum=h_q, H_classical=h_c,
                params=sum(p.numel() for p in model.parameters() if p.requires_grad),
                n_qubits=cfg["n_qubits"], n_heads=cfg["n_heads"],
                seconds=round(time.time() - t0, 1))


def mean_std(xs):
    n = len(xs)
    mu = sum(xs) / n
    sd = (sum((x - mu) ** 2 for x in xs) / (n - 1)) ** 0.5 if n > 1 else 0.0
    return mu, sd


def permutation_test(a, b, max_exact=200000, n_random=100000, seed=0):
    a, b = list(a), list(b)
    na, nb = len(a), len(b)
    pooled = a + b
    obs = abs(sum(a) / na - sum(b) / nb)
    total = math.comb(na + nb, na)
    cnt = 0
    if total <= max_exact:
        for combo in itertools.combinations(range(na + nb), na):
            s = set(combo)
            ga = [pooled[i] for i in combo]
            gb = [pooled[i] for i in range(na + nb) if i not in s]
            if abs(sum(ga) / na - sum(gb) / nb) >= obs - 1e-12:
                cnt += 1
        return obs, cnt / total, f"exact ({total})"
    rng = torch.Generator().manual_seed(seed)
    for _ in range(n_random):
        perm = torch.randperm(na + nb, generator=rng).tolist()
        ga = [pooled[i] for i in perm[:na]]
        gb = [pooled[i] for i in perm[na:]]
        if abs(sum(ga) / na - sum(gb) / nb) >= obs - 1e-12:
            cnt += 1
    return obs, cnt / n_random, f"sampled ({n_random})"


## 4 · Dataset config + load + fixed split

In [ ]:

import csv as _csv
def load_dataset_examples():
    def rd(p):
        ex=[]
        with open(p,encoding="utf-8",newline="") as f:
            for row in _csv.reader(f):
                if len(row)<3 or not row[0].strip().isdigit(): continue
                t=(row[1]+" "+row[2]).replace("\\"," ").strip()
                if t: ex.append({"text":t,"label":int(row[0])-1})
        return ex
    return rd("agnews/train.csv"), rd("agnews/test.csv"), ["World","Sports","Business","Sci/Tech"]

# ---- experiment configuration ----
N_CLASSES   = 4
MAX_LEN     = 53
DEFAULT_N   = 5000      # train-fit size for experiments A, B, D
EPOCHS      = 20
VAL_SIZE    = 500
MIN_FREQ    = 2
SEEDS       = [0, 1, 2, 3, 4]       # set to [0,1,2] for a quick run
N_EFF       = [250, 500, 1000, 2000, 5000]          # training sizes for the data-efficiency sweep

TRAIN_ALL, TEST_ALL, LABELS = load_dataset_examples()
assert len(LABELS) == N_CLASSES
print(f"{DATASET}: train={len(TRAIN_ALL)} test={len(TEST_ALL)} classes={N_CLASSES} {LABELS}")
from collections import Counter
print("label counts:", dict(Counter(e['label'] for e in TRAIN_ALL)))
_lens = sorted(len(tokenize(e['text'])) for e in TRAIN_ALL)
print(f"token length: median={_lens[len(_lens)//2]} p95={_lens[int(0.95*len(_lens))]} "
      f"max={_lens[-1]} | MAX_LEN={MAX_LEN}")


agnews: train=120000 test=7600 classes=4 ['World', 'Sports', 'Business', 'Sci/Tech']
label counts: {2: 30000, 3: 30000, 1: 30000, 0: 30000}
token length: median=37 p95=53 max=177 | MAX_LEN=53


## 5 · Checkpointed experiment harness
`run_experiment` saves to Drive after **every** (config, seed); re-running skips finished cells. Data/vocab are built per training size and cached.

In [ ]:
def _load(path):
    return json.load(open(path)) if os.path.exists(path) else {}

def _save(path, obj):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    tmp = path + ".tmp"
    json.dump(obj, open(tmp, "w"), indent=2)
    os.replace(tmp, path)            # avoids half-written checkpoints on interrupt

_data_cache = {}
def make_split(limit_train):
    g = torch.Generator().manual_seed(12345)
    tr = list(TRAIN_ALL)
    perm = torch.randperm(len(tr), generator=g).tolist()
    tr = [tr[i] for i in perm]
    if limit_train is not None:
        tr = tr[:limit_train + VAL_SIZE]
    val = tr[:VAL_SIZE]; train_fit = tr[VAL_SIZE:]
    vocab = build_vocab(train_fit, MIN_FREQ)
    return (train_fit, val, TEST_ALL), vocab, N_CLASSES

def get_data(limit_train):
    if limit_train not in _data_cache:
        _data_cache[limit_train] = make_split(limit_train)
    return _data_cache[limit_train]

BASE = dict(epochs=EPOCHS, max_len=MAX_LEN, batch_size=64, d_model=64,
            n_heads=4, n_qubits=4, lr=2e-3, qlr=1e-2, device=DEVICE)
def cfg(cond, key, **over):
    c = dict(BASE); c.update(over); c["cond"] = cond; c["key"] = key
    c.setdefault("limit_train", DEFAULT_N); return c

def run_experiment(exp_name, grid, seeds):
    path = os.path.join(RESULTS_DIR, exp_name + ".json")
    results = _load(path)
    t_start = time.time()
    for c in grid:
        key = c["key"]
        data, vocab, nc = get_data(c["limit_train"])
        results.setdefault(key, {})
        for s in seeds:
            if str(s) in results[key]:
                print(f"  skip {key} seed{s} (test={results[key][str(s)]['test_acc']:.4f})")
                continue
            print(f"  run  {key} seed{s} ...", flush=True)
            r = run_one(c["cond"], s, data, vocab, nc, c)
            results[key][str(s)] = r
            _save(path, results)
            print(f"       test={r['test_acc']:.4f} val={r['val_acc']:.4f} "
                  f"ep*={r['best_epoch']} H_q={r['H_quantum']:.3f} "
                  f"H_c={r['H_classical']:.3f} ({r['seconds']}s)", flush=True)
    print(f"[done] {exp_name}: {time.time()-t_start:.0f}s total -> {path}")
    return results


## Circuit Analysis — Separability and CZ Visibility (no training)
These analytic checks ground the paper's methodology (Table 2, Figure 2). They run in seconds and need no training or data:

1. **Separability** — for `none`/`intra`, the score factorises (⟨Z₀Z₂⟩ = ⟨Z₀⟩⟨Z₂⟩); for `cross`/`full` it does not. This is what lets the paper attribute any effect specifically to query–key entanglement.
2. **CZ visibility** — without the post-entangler rotation layer, the entangling gates are *invisible* to the Z-basis readout (Δ⟨Z₀Z₂⟩ = 0). The post-rotation layer makes them measurable. This is the corrected-ansatz contribution.
3. **Entanglement entropy** — von Neumann entropy across the query|key cut confirms `cross`/`full` produce genuine entanglement; `none`/`intra` produce none.

In [ ]:
import numpy as np
import matplotlib
matplotlib.use("Agg"); import matplotlib.pyplot as plt

def _analysis_qnodes(n_qubits=4, with_post=True):
    h = n_qubits // 2
    qa = list(range(h)); ka = list(range(h, n_qubits))
    cross = [(qa[i], ka[i]) for i in range(h)]
    intra = ([(qa[i], qa[i+1]) for i in range(h-1)]
             + [(ka[i], ka[i+1]) for i in range(h-1)])
    modes = {"none": [], "intra": intra, "cross": cross, "full": cross + intra}
    dev = qml.device("default.qubit", wires=n_qubits)
    def _body(inp, w, cz):
        qml.AngleEmbedding(inp, wires=range(n_qubits), rotation="Y")
        for q in range(n_qubits):
            qml.RX(w[0,q,0], q); qml.RY(w[0,q,1], q); qml.RZ(w[0,q,2], q)
        for a, b in cz: qml.CZ([a, b])
        if with_post:
            for q in range(n_qubits):
                qml.RX(w[1,q,0], q); qml.RY(w[1,q,1], q); qml.RZ(w[1,q,2], q)
    out = {}
    for m, cz in modes.items():
        @qml.qnode(dev)
        def ev(inp, w, cz=cz):
            _body(inp, w, cz)
            return [qml.expval(qml.PauliZ(qa[0])), qml.expval(qml.PauliZ(ka[0])),
                    qml.expval(qml.PauliZ(qa[0]) @ qml.PauliZ(ka[0]))]
        @qml.qnode(dev)
        def vn(inp, w, cz=cz):
            _body(inp, w, cz)
            return qml.vn_entropy(wires=qa)
        out[m] = (ev, vn)
    return out

rng = np.random.default_rng(0)
NQ = 4; N_SAMPLES = 300
modes = ["none", "intra", "cross", "full"]
qn  = _analysis_qnodes(NQ, with_post=True)
qn0 = _analysis_qnodes(NQ, with_post=False)

rows = {}
print("=== Table 2: separability + query|key entanglement (post-rotation ON) ===")
print(f"{'mode':6} {'|<Z0Z2>-<Z0><Z2>|':>20} {'separable':>10} {'VN_entropy':>12}")
for m in modes:
    d, e = [], []
    for _ in range(N_SAMPLES):
        x = rng.uniform(-np.pi, np.pi, NQ); w = rng.uniform(-np.pi, np.pi, (2, NQ, 3))
        z0, z2, z0z2 = qn[m][0](x, w)
        d.append(abs(z0z2 - z0 * z2)); e.append(float(qn[m][1](x, w)))
    rows[m] = (np.mean(d), np.mean(e))
    print(f"{m:6} {np.mean(d):20.6f} {('YES' if np.mean(d) < 1e-9 else 'no'):>10} {np.mean(e):12.4f}")

print("\n=== Figure 2: entangler effect on <Z0Z2> with vs without post-rotation ===")
vis = {"with": {}, "without": {}}
for m in modes:
    for tag, q in (("with", qn), ("without", qn0)):
        diffs = []
        for _ in range(N_SAMPLES):
            x = rng.uniform(-np.pi, np.pi, NQ); w = rng.uniform(-np.pi, np.pi, (2, NQ, 3))
            on = q[m][0](x, w)[2]; off = q["none"][0](x, w)[2]
            diffs.append(abs(on - off))
        vis[tag][m] = float(np.mean(diffs))
    print(f"  {m:6} | with post-rot Δ={vis['with'][m]:.4f}   "
          f"without post-rot Δ={vis['without'][m]:.4f}")

# save CSV
import csv
csv_path = os.path.join(RESULTS_DIR, "circuit_analysis.csv")
with open(csv_path, "w", newline="") as f:
    wri = csv.writer(f)
    wri.writerow(["mode", "sep_gap", "vn_entropy",
                  "delta_with_postrot", "delta_without_postrot"])
    for m in modes:
        wri.writerow([m, rows[m][0], rows[m][1], vis["with"][m], vis["without"][m]])
print("wrote", csv_path)

# figure: CZ visibility (grouped bars)
fig, ax = plt.subplots(figsize=(6.4, 4.2))
x = np.arange(len(modes)); bw = 0.38
ax.bar(x - bw/2, [vis["with"][m] for m in modes], bw, label="with post-rotation",
       color="#0a3d62", edgecolor="black", linewidth=.6)
ax.bar(x + bw/2, [vis["without"][m] for m in modes], bw, label="without post-rotation",
       color="#b0b0b0", edgecolor="black", linewidth=.6)
ax.set_xticks(x); ax.set_xticklabels(modes)
ax.set_ylabel(r"$|\Delta\langle Z_0 Z_2\rangle|$ vs none")
ax.set_title(f"{DATASET.upper()} · entangler visibility (n_qubits={NQ})")
ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
for ext in ("png", "pdf"):
    fig.savefig(os.path.join(FIG_DIR, f"circuit_visibility.{ext}"), dpi=300, bbox_inches="tight")
plt.close(fig)
print("saved circuit_visibility figure to", FIG_DIR)


=== Table 2: separability + query|key entanglement (post-rotation ON) ===
mode      |<Z0Z2>-<Z0><Z2>|  separable   VN_entropy
none               0.000000        YES       0.0000
intra              0.000000        YES       0.0000
cross              0.256676         no       0.6586
full               0.105316         no       0.6388

=== Figure 2: entangler effect on <Z0Z2> with vs without post-rotation ===
  none   | with post-rot Δ=0.0000   without post-rot Δ=0.0000
  intra  | with post-rot Δ=0.2640   without post-rot Δ=0.0000
  cross  | with post-rot Δ=0.4172   without post-rot Δ=0.0000
  full   | with post-rot Δ=0.2721   without post-rot Δ=0.0000
wrote /content/drive/MyDrive/quantum_attention/agnews/circuit_analysis.csv
saved circuit_visibility figure to /content/drive/MyDrive/quantum_attention/agnews/figures


## Experiment A — Architecture ablation (4 heads)
The quantum head is **1 of 4** attention heads (the realistic setting). All five conditions share the same architecture; only the entangling-gate set changes.

In [ ]:
gridA = [cfg(c, f"A_{c}") for c in
         ["classical","qnone","qintra","qcross","qfull"]]
run_experiment("expA_4head", gridA, SEEDS)


  skip A_classical seed0 (test=0.7742)
  skip A_classical seed1 (test=0.7701)
  skip A_classical seed2 (test=0.7670)
  skip A_classical seed3 (test=0.7738)
  skip A_classical seed4 (test=0.7691)
  skip A_qnone seed0 (test=0.7617)
  skip A_qnone seed1 (test=0.7814)
  skip A_qnone seed2 (test=0.7633)
  skip A_qnone seed3 (test=0.7739)
  skip A_qnone seed4 (test=0.7747)
  skip A_qintra seed0 (test=0.7692)
  skip A_qintra seed1 (test=0.7675)
  skip A_qintra seed2 (test=0.7614)
  skip A_qintra seed3 (test=0.7532)
  skip A_qintra seed4 (test=0.7562)
  skip A_qcross seed0 (test=0.7629)
  skip A_qcross seed1 (test=0.7749)
  skip A_qcross seed2 (test=0.7747)
  skip A_qcross seed3 (test=0.7626)
  skip A_qcross seed4 (test=0.7779)
  skip A_qfull seed0 (test=0.7700)
  skip A_qfull seed1 (test=0.7741)
  skip A_qfull seed2 (test=0.7701)
  skip A_qfull seed3 (test=0.7729)
  skip A_qfull seed4 (test=0.7428)
[done] expA_4head: 0s total -> /content/drive/MyDrive/quantum_attention/agnews/expA_4head.json


{'A_classical': {'0': {'test_acc': 0.7742105263157895,
   'val_acc': 0.806,
   'best_epoch': 16,
   'H_quantum': nan,
   'H_classical': 2.146724581718445,
   'params': 856196,
   'n_qubits': 4,
   'n_heads': 4,
   'seconds': 106.3},
  '1': {'test_acc': 0.7701315789473684,
   'val_acc': 0.782,
   'best_epoch': 18,
   'H_quantum': nan,
   'H_classical': 2.0709230105082193,
   'params': 856196,
   'n_qubits': 4,
   'n_heads': 4,
   'seconds': 104.2},
  '2': {'test_acc': 0.7669736842105264,
   'val_acc': 0.784,
   'best_epoch': 16,
   'H_quantum': nan,
   'H_classical': 2.4370965162913003,
   'params': 856196,
   'n_qubits': 4,
   'n_heads': 4,
   'seconds': 105.9},
  '3': {'test_acc': 0.7738157894736842,
   'val_acc': 0.772,
   'best_epoch': 16,
   'H_quantum': nan,
   'H_classical': 2.219522476196289,
   'params': 856196,
   'n_qubits': 4,
   'n_heads': 4,
   'seconds': 105.4},
  '4': {'test_acc': 0.7690789473684211,
   'val_acc': 0.772,
   'best_epoch': 17,
   'H_quantum': nan,
   'H_cl

## Experiment B — Single-head ablation (dilution test)
`n_heads=1`: the quantum circuit is the **only** attention mechanism. If entanglement matters at all, it should be most visible here, undiluted by parallel classical heads.

In [ ]:
gridB = [cfg(c, f"B_{c}", n_heads=1) for c in
         ["classical","qnone","qintra","qcross","qfull"]]
run_experiment("expB_1head", gridB, SEEDS)


  skip B_classical seed0 (test=0.7789)
  skip B_classical seed1 (test=0.7770)
  skip B_classical seed2 (test=0.7768)
  skip B_classical seed3 (test=0.7758)
  skip B_classical seed4 (test=0.7789)
  skip B_qnone seed0 (test=0.7854)
  skip B_qnone seed1 (test=0.7899)
  skip B_qnone seed2 (test=0.7803)
  skip B_qnone seed3 (test=0.7824)
  skip B_qnone seed4 (test=0.7849)
  skip B_qintra seed0 (test=0.7808)
  skip B_qintra seed1 (test=0.7916)
  skip B_qintra seed2 (test=0.7916)
  skip B_qintra seed3 (test=0.7718)
  skip B_qintra seed4 (test=0.7770)
  skip B_qcross seed0 (test=0.7822)
  skip B_qcross seed1 (test=0.7911)
  skip B_qcross seed2 (test=0.7886)
  skip B_qcross seed3 (test=0.7691)
  skip B_qcross seed4 (test=0.7813)
  skip B_qfull seed0 (test=0.7813)
  skip B_qfull seed1 (test=0.7871)
  skip B_qfull seed2 (test=0.7786)
  skip B_qfull seed3 (test=0.7739)
  skip B_qfull seed4 (test=0.7661)
[done] expB_1head: 0s total -> /content/drive/MyDrive/quantum_attention/agnews/expB_1head.json


{'B_classical': {'0': {'test_acc': 0.7789473684210526,
   'val_acc': 0.788,
   'best_epoch': 16,
   'H_quantum': nan,
   'H_classical': 2.051714857419332,
   'params': 856388,
   'n_qubits': 4,
   'n_heads': 1,
   'seconds': 71.1},
  '1': {'test_acc': 0.7769736842105263,
   'val_acc': 0.8,
   'best_epoch': 16,
   'H_quantum': nan,
   'H_classical': 2.392922878265381,
   'params': 856388,
   'n_qubits': 4,
   'n_heads': 1,
   'seconds': 72.4},
  '2': {'test_acc': 0.7768421052631579,
   'val_acc': 0.8,
   'best_epoch': 16,
   'H_quantum': nan,
   'H_classical': 2.4929540157318115,
   'params': 856388,
   'n_qubits': 4,
   'n_heads': 1,
   'seconds': 73.8},
  '3': {'test_acc': 0.7757894736842105,
   'val_acc': 0.776,
   'best_epoch': 18,
   'H_quantum': nan,
   'H_classical': 2.3283743460973105,
   'params': 856388,
   'n_qubits': 4,
   'n_heads': 1,
   'seconds': 71.0},
  '4': {'test_acc': 0.7789473684210526,
   'val_acc': 0.772,
   'best_epoch': 16,
   'H_quantum': nan,
   'H_classical'

## Experiment C — Data-efficiency
Accuracy vs training size. Quantum methods are most defensible in the low-data regime, so this curve is where a positive effect is most likely to appear.

In [ ]:
gridC = [cfg(c, f"C_{c}_N{N}", limit_train=N)
         for N in N_EFF
         for c in ["classical","qnone","qcross","qfull"]]
run_experiment("expC_dataeff", gridC, SEEDS)


  skip C_classical_N250 seed0 (test=0.3887)
  skip C_classical_N250 seed1 (test=0.4397)
  skip C_classical_N250 seed2 (test=0.4616)
  skip C_classical_N250 seed3 (test=0.4567)
  skip C_classical_N250 seed4 (test=0.4282)
  skip C_qnone_N250 seed0 (test=0.3812)
  skip C_qnone_N250 seed1 (test=0.4154)
  skip C_qnone_N250 seed2 (test=0.4399)
  skip C_qnone_N250 seed3 (test=0.4279)
  skip C_qnone_N250 seed4 (test=0.4276)
  skip C_qcross_N250 seed0 (test=0.3851)
  skip C_qcross_N250 seed1 (test=0.4145)
  skip C_qcross_N250 seed2 (test=0.4217)
  skip C_qcross_N250 seed3 (test=0.4311)
  skip C_qcross_N250 seed4 (test=0.4109)
  skip C_qfull_N250 seed0 (test=0.3914)
  skip C_qfull_N250 seed1 (test=0.4441)
  skip C_qfull_N250 seed2 (test=0.4496)
  skip C_qfull_N250 seed3 (test=0.4357)
  skip C_qfull_N250 seed4 (test=0.4191)
  skip C_classical_N500 seed0 (test=0.5391)
  skip C_classical_N500 seed1 (test=0.4997)
  skip C_classical_N500 seed2 (test=0.5403)
  skip C_classical_N500 seed3 (test=0.5100)

## Experiment D — Qubit width / encoding bottleneck
2 / 4 / 8 qubits = 1 / 2 / 4 angles encoded per side. Tests whether widening the input encoding gives entanglement room to matter. **Memory note:** the statevector is 2^n_qubits per pair, so this cell uses a smaller batch and sequence length; n_qubits=8 is the slowest run in the notebook.

In [ ]:
# Conservative settings keep n_qubits=8 within memory; classical is the reference.
# n_qubits=8 is the slowest cell in the whole study (statevector = 2^8 per pair),
# so this experiment uses a smaller subset, fewer epochs, and fewer seeds.
# The grid runs nq=2 and nq=4 first (cheap) and checkpoints, so you get those
# results before nq=8 starts — you can interrupt before nq=8 if short on time.
D_OVER  = dict(batch_size=16, max_len=min(MAX_LEN, 14), limit_train=800, epochs=12)
D_SEEDS = SEEDS[:3]                         # 3 seeds for the expensive sweep
gridD = [cfg("classical", "D_classical", **D_OVER)]
for nq in [2, 4, 8]:                        # nq=8 last (slowest)
    for c in ["qnone","qcross","qfull"]:
        gridD.append(cfg(c, f"D_{c}_nq{nq}", n_qubits=nq, **D_OVER))
run_experiment("expD_qubits", gridD, D_SEEDS)


## Reporting & figures
Reads the Drive checkpoints, prints mean±std tables with permutation tests, and saves publication figures (PNG + PDF) to the Drive `figures/` folder. Safe to run with partial results.

In [ ]:
import matplotlib
matplotlib.use("Agg"); import matplotlib.pyplot as plt
plt.rcParams.update({"font.size":11,"axes.spines.top":False,"axes.spines.right":False,
                     "axes.grid":True,"grid.alpha":0.25,"grid.linestyle":"--"})

def _accs(res, key):
    return [v["test_acc"] for v in res.get(key, {}).values()]

def _ms(xs):
    n=len(xs); mu=sum(xs)/n if n else float('nan')
    sd=(sum((x-mu)**2 for x in xs)/(n-1))**0.5 if n>1 else 0.0
    return mu,sd,n

def table(res, keys, title, pairs=None):
    print(f"\n=== {title} ===")
    print(f"{'key':16} {'mean':>8} {'std':>7} {'n':>3}  H_q     H_c")
    summ={}
    for k in keys:
        a=_accs(res,k)
        if not a: continue
        mu,sd,n=_ms(a); summ[k]=a
        runs=list(res[k].values())
        hq=sum(float(x['H_quantum']) if str(x['H_quantum'])!='nan' else float('nan') for x in runs)/len(runs)
        hc=sum(float(x['H_classical']) for x in runs)/len(runs)
        print(f"{k:16} {mu:8.4f} {sd:7.4f} {n:3d}  {hq:6.3f}  {hc:6.3f}")
    print(f"{'random':16} {1.0/N_CLASSES:8.4f}")
    if pairs:
        for x,y in pairs:
            if len(summ.get(x,[]))>1 and len(summ.get(y,[]))>1:
                _,p,mode=permutation_test(summ[x],summ[y])
                dm=sum(summ[x])/len(summ[x])-sum(summ[y])/len(summ[y])
                print(f"   {x} vs {y}: Δ={dm:+.4f} p={p:.4f} {'*' if p<0.05 else ''} [{mode}]")
    return summ

def bar(res, keys, labels, title, fname):
    ms=[_ms(_accs(res,k)) for k in keys]
    if not any(m[2] for m in ms): print("no data for",fname); return
    mu=[m[0] for m in ms]; sd=[m[1] for m in ms]
    cols=["#9aa0a6","#a8d0e6","#a8d0e6","#1b6ca8","#0a3d62"][:len(keys)]
    fig,ax=plt.subplots(figsize=(6.2,4.2)); x=range(len(keys))
    ax.bar(x,mu,yerr=sd,capsize=5,color=cols,edgecolor="black",linewidth=.6)
    ax.set_xticks(list(x)); ax.set_xticklabels(labels)
    ax.set_ylabel("test accuracy"); ax.set_title(title)
    lo=min(m-s for m,s in zip(mu,sd)); hi=max(m+s for m,s in zip(mu,sd))
    pad=(hi-lo)*.5+.005; ax.set_ylim(lo-pad,hi+pad*1.4)
    for xi,m in zip(x,mu): ax.text(xi,m,f"{m:.3f}",ha="center",va="bottom",fontsize=8)
    fig.tight_layout()
    for ext in ("png","pdf"): fig.savefig(os.path.join(FIG_DIR,f"{fname}.{ext}"),dpi=300,bbox_inches="tight")
    plt.close(fig); print("saved",fname)

def entropy_bar(res, keys, labels, title, fname):
    # H_q vs H_c per condition (the robust inductive-bias finding)
    def _h(k, field):
        runs=list(res.get(k,{}).values())
        vals=[float(r[field]) for r in runs if str(r[field])!='nan']
        if not vals: return float('nan'), 0.0
        mu=sum(vals)/len(vals)
        sd=(sum((v-mu)**2 for v in vals)/(len(vals)-1))**0.5 if len(vals)>1 else 0.0
        return mu, sd
    hq=[_h(k,'H_quantum') for k in keys]; hc=[_h(k,'H_classical') for k in keys]
    if not any(v[0]==v[0] for v in hq+hc): print("no entropy data for",fname); return
    import numpy as _np
    x=_np.arange(len(keys)); bw=0.38
    fig,ax=plt.subplots(figsize=(6.4,4.2))
    ax.bar(x-bw/2,[v[0] for v in hq],bw,yerr=[v[1] for v in hq],capsize=4,
           label="quantum head $H_q$",color="#0a3d62",edgecolor="black",linewidth=.6)
    ax.bar(x+bw/2,[v[0] for v in hc],bw,yerr=[v[1] for v in hc],capsize=4,
           label="classical head $H_c$",color="#b0b0b0",edgecolor="black",linewidth=.6)
    ax.set_xticks(x); ax.set_xticklabels(labels)
    ax.set_ylabel("attention entropy (nats)"); ax.set_title(title)
    ax.legend(frameon=False)
    fig.tight_layout()
    for ext in ("png","pdf"): fig.savefig(os.path.join(FIG_DIR,f"{fname}.{ext}"),dpi=300,bbox_inches="tight")
    plt.close(fig); print("saved",fname)

def line_vs(res, xvals, key_fmt, conds, title, xlabel, fname):
    fig,ax=plt.subplots(figsize=(6.2,4.2))
    for c in conds:
        ys,es=[],[]
        for xv in xvals:
            mu,sd,n=_ms(_accs(res,key_fmt(c,xv))); ys.append(mu); es.append(sd)
        if any(y==y for y in ys):
            ax.errorbar(xvals,ys,yerr=es,marker="o",capsize=4,label=c)
    ax.set_xlabel(xlabel); ax.set_ylabel("test accuracy"); ax.set_title(title)
    ax.axhline(1.0/N_CLASSES,ls=":",color="#888",label="random")
    ax.legend(frameon=False,fontsize=9); fig.tight_layout()
    for ext in ("png","pdf"): fig.savefig(os.path.join(FIG_DIR,f"{fname}.{ext}"),dpi=300,bbox_inches="tight")
    plt.close(fig); print("saved",fname)

CONDS5=["classical","qnone","qintra","qcross","qfull"]
LAB5=["classical","q-none","q-intra","q-cross","q-full"]
ABL_PAIRS=[("{p}qfull","{p}classical"),("{p}qfull","{p}qnone"),
           ("{p}qcross","{p}qnone"),("{p}qfull","{p}qcross")]

# Experiment A
rA=_load(os.path.join(RESULTS_DIR,"expA_4head.json"))
if rA:
    table(rA,[f"A_{c}" for c in CONDS5],"A · 4-head ablation",
          [(x.format(p="A_"),y.format(p="A_")) for x,y in ABL_PAIRS])
    bar(rA,[f"A_{c}" for c in CONDS5],LAB5,f"{DATASET.upper()} · 4-head ablation","A_4head")
    entropy_bar(rA,[f"A_{c}" for c in ["qnone","qintra","qcross","qfull"]],
                ["q-none","q-intra","q-cross","q-full"],
                f"{DATASET.upper()} · attention entropy (4-head)","A_entropy")

# Experiment B
rB=_load(os.path.join(RESULTS_DIR,"expB_1head.json"))
if rB:
    table(rB,[f"B_{c}" for c in CONDS5],"B · single-head ablation",
          [(x.format(p="B_"),y.format(p="B_")) for x,y in ABL_PAIRS])
    bar(rB,[f"B_{c}" for c in CONDS5],LAB5,f"{DATASET.upper()} · single head","B_1head")

# Experiment C
rC=_load(os.path.join(RESULTS_DIR,"expC_dataeff.json"))
if rC:
    table(rC,[f"C_{c}_N{N}" for N in N_EFF for c in ["classical","qnone","qcross","qfull"]],
          "C · data-efficiency")
    line_vs(rC,N_EFF,lambda c,N:f"C_{c}_N{N}",["classical","qnone","qcross","qfull"],
            f"{DATASET.upper()} · data efficiency","train size N","C_dataeff")

# Experiment D
rD=_load(os.path.join(RESULTS_DIR,"expD_qubits.json"))
if rD:
    table(rD,["D_classical"]+[f"D_{c}_nq{nq}" for nq in [2,4,8] for c in ["qnone","qcross","qfull"]],
          "D · qubit width")
    line_vs(rD,[2,4,8],lambda c,nq:f"D_{c}_nq{nq}",["qnone","qcross","qfull"],
            f"{DATASET.upper()} · qubit width","n_qubits","D_qubits")

print("\nAll figures saved to", FIG_DIR)


In [ ]:
gridE = [cfg(c, f"E_{c}", quantum_in_block=1) for c in
         ["classical","qnone","qintra","qcross","qfull"]]
run_experiment("expE_depth", gridE, SEEDS)

In [ ]:
# Experiment E — depth ablation reporting
rE=_load(os.path.join(RESULTS_DIR,"expE_depth.json"))
if rE:
    table(rE,[f"E_{c}" for c in CONDS5],"E · quantum head in LAST block",
          [(x.format(p="E_"),y.format(p="E_")) for x,y in ABL_PAIRS])
    bar(rE,[f"E_{c}" for c in CONDS5],LAB5,f"{DATASET.upper()} · quantum in last block","E_depth")

# Cross-experiment: does layer position move accuracy?  (A = block 0, E = block 1)
rA_=_load(os.path.join(ORIG_DIR,"expA_4head.json"))
if rA_ and rE:
    print("\n=== Layer-position effect: block 0 (A) vs block 1 (E) ===")
    for c in CONDS5:
        a=_accs(rA_,f"A_{c}"); e=_accs(rE,f"E_{c}")
        if len(a)>1 and len(e)>1:
            _,p,mode=permutation_test(a,e)
            d=sum(e)/len(e)-sum(a)/len(a)
            print(f"   {c:10} block1-block0 Δ={d:+.4f} p={p:.4f} {'*' if p<0.05 else ''} [{mode}]")